# LLM Workflows with LangChain (Python)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RPI-WS-spring-2026/Agents1/blob/main/python-workflow/workflow_example.ipynb)

In this notebook, you'll build a multi-step AI workflow — a pipeline where each step transforms data or calls an LLM, and the output of one step feeds the next.

**What we'll build:** A content generation workflow that takes a topic and produces:
1. An outline
2. A draft based on the outline
3. A summary of the draft

Along the way, we'll handle errors that can occur at each step.

## Setup

First, install dependencies and configure your API key.

You'll need a **free Groq API key**:
1. Go to [console.groq.com](https://console.groq.com) and sign up (free)
2. Go to **API Keys** and create a new key

**On Google Colab:** After running the install cell below:
1. Click the **key icon** (🔑) in the left sidebar
2. Add a new secret named `GROQ_API_KEY` with your key as the value
3. Toggle **"Notebook access"** on

**On Codespaces / local:** Set the environment variable: `export GROQ_API_KEY="your-key-here"` or create a `.env` file.

In [ ]:
# Install dependencies (run once)
!pip install langchain langchain-groq python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv

# Load .env file if it exists (for Codespaces / local)
load_dotenv()

# If running on Google Colab, pull the key from Colab Secrets
if not os.getenv("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        print("API key loaded from Colab Secrets.")
    except (ImportError, userdata.SecretNotFoundError):
        pass  # Not on Colab or secret not set

# Verify the key is set
if not os.getenv("GROQ_API_KEY"):
    print("WARNING: GROQ_API_KEY is not set!")
    print("  1. Get a free key at https://console.groq.com")
    print("  Colab: Click the key icon in the left sidebar, add GROQ_API_KEY")
    print("  Local: export GROQ_API_KEY='your-key-here'")
else:
    print("API key is configured.")

## Step 1: A Simple Chain

A **chain** in LangChain connects a prompt template to an LLM. The prompt template has placeholders that get filled in at runtime.

LangChain uses the `|` (pipe) operator to connect components — just like Unix pipes.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize the LLM — using Groq (free, ultra-fast inference)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.7,
)

# Create a simple chain: prompt -> LLM -> parse output as string
simple_chain = (
    ChatPromptTemplate.from_template("Tell me one interesting fact about {topic}.")
    | llm
    | StrOutputParser()
)

# Run it
result = simple_chain.invoke({"topic": "the history of the internet"})
print(result)

## Step 2: Multi-Step Workflow

Now let's build a real workflow with 3 steps:

```
Topic → [Generate Outline] → [Write Draft] → [Create Summary] → Final Output
```

Each step is its own chain. We'll connect them using LangChain's `RunnablePassthrough` and `RunnableLambda` to pass data between steps.

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Step 1: Generate an outline from a topic
outline_prompt = ChatPromptTemplate.from_template(
    """Create a brief outline for a blog post about: {topic}

The outline should have 3-4 main sections with 2-3 bullet points each.
Format it as a numbered list."""
)

outline_chain = outline_prompt | llm | StrOutputParser()

# Step 2: Write a draft based on the outline
draft_prompt = ChatPromptTemplate.from_template(
    """Write a short blog post (about 300 words) based on this outline:

{outline}

Make it engaging and informative. Use a conversational tone."""
)

draft_chain = draft_prompt | llm | StrOutputParser()

# Step 3: Summarize the draft
summary_prompt = ChatPromptTemplate.from_template(
    """Summarize this blog post in 2-3 sentences:

{draft}"""
)

summary_chain = summary_prompt | llm | StrOutputParser()

print("Chains defined. Ready to run the workflow.")

In [ ]:
# Run the full workflow step by step so we can see intermediate results
topic = "how AI is changing web development"

print("=" * 60)
print("STEP 1: Generating outline...")
print("=" * 60)
outline = outline_chain.invoke({"topic": topic})
print(outline)

print("\n" + "=" * 60)
print("STEP 2: Writing draft...")
print("=" * 60)
draft = draft_chain.invoke({"outline": outline})
print(draft)

print("\n" + "=" * 60)
print("STEP 3: Creating summary...")
print("=" * 60)
summary = summary_chain.invoke({"draft": draft})
print(summary)

## Step 3: Composing Into a Single Chain

We can also compose the entire workflow into a single chain using `RunnableLambda` to pass data between steps.

In [ ]:
# Compose the full pipeline into one chain
full_workflow = (
    # Start with the topic, generate an outline
    {"outline": outline_chain}
    # Feed the outline into the draft chain
    | draft_chain
    # We need to restructure for the summary chain
    | RunnableLambda(lambda draft: {"draft": draft})
    | summary_chain
)

# Run the entire workflow with one call
final_summary = full_workflow.invoke({"topic": "the future of serverless computing"})
print("Final summary:")
print(final_summary)

## Step 4: Error Handling

LLM workflows can fail in many ways. Let's build error handling into our workflow.

### 4a: Handling API Errors (Rate Limits, Timeouts)

In [ ]:
import time


def call_with_retry(chain, inputs, max_retries=3):
    """Call a chain with retry logic for transient errors."""
    for attempt in range(1, max_retries + 1):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            error_msg = str(e)
            # Check if it's a retryable error (rate limit or server error)
            if "429" in error_msg or "500" in error_msg or "timeout" in error_msg.lower():
                if attempt < max_retries:
                    wait_time = 2 ** attempt  # Exponential backoff: 2, 4, 8 seconds
                    print(f"Attempt {attempt} failed ({error_msg[:50]}...). "
                          f"Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"All {max_retries} attempts failed.")
                    raise
            else:
                # Non-retryable error (bad request, auth error, etc.)
                print(f"Non-retryable error: {error_msg[:100]}")
                raise


# Example: use retry logic in the workflow
outline = call_with_retry(outline_chain, {"topic": "quantum computing basics"})
print("Outline generated successfully!")
print(outline[:200] + "...")

### 4b: Validating LLM Output

LLMs can return unexpected formats. Let's validate that we get what we expect.

In [ ]:
import json


def validate_json_output(llm_output, required_keys):
    """Validate that LLM output is valid JSON with required keys."""
    # Try to extract JSON from the output (LLMs sometimes add extra text)
    text = llm_output.strip()

    # Look for JSON block in markdown code fences
    if "```json" in text:
        start = text.index("```json") + 7
        end = text.index("```", start)
        text = text[start:end].strip()
    elif "```" in text:
        start = text.index("```") + 3
        end = text.index("```", start)
        text = text[start:end].strip()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"LLM returned invalid JSON: {e}\n"
            f"Raw output (first 200 chars): {llm_output[:200]}"
        )

    missing = [k for k in required_keys if k not in parsed]
    if missing:
        raise ValueError(f"Missing required keys in LLM output: {missing}")

    return parsed


# Create a chain that requests structured JSON output
json_prompt = ChatPromptTemplate.from_template(
    """Analyze the following topic and return a JSON object with these exact keys:
- "title": a catchy title (string)
- "tags": relevant tags (array of strings, 3-5 tags)
- "difficulty": how hard this topic is to explain ("beginner", "intermediate", or "advanced")

Topic: {topic}

Return ONLY valid JSON, no other text."""
)

json_chain = json_prompt | llm | StrOutputParser()

# Run and validate
raw_output = json_chain.invoke({"topic": "CSS Grid Layout"})
print("Raw LLM output:")
print(raw_output)
print()

# Validate the output
try:
    parsed = validate_json_output(raw_output, ["title", "tags", "difficulty"])
    print("Validated output:")
    print(json.dumps(parsed, indent=2))
except ValueError as e:
    print(f"Validation failed: {e}")

### 4c: Fallback Chains

LangChain has built-in support for fallback chains — if the primary model fails, automatically try a backup.

In [ ]:
# Create a primary model and a fallback model
primary_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.7,
)
fallback_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.3,  # Smaller, faster model as fallback
)

# Use LangChain's built-in .with_fallbacks() method
llm_with_fallback = primary_llm.with_fallbacks([fallback_llm])

# Build a chain using the fallback-enabled LLM
robust_chain = (
    ChatPromptTemplate.from_template("Explain {topic} in simple terms.")
    | llm_with_fallback
    | StrOutputParser()
)

result = robust_chain.invoke({"topic": "WebSockets"})
print(result)

### 4c-2: Tool Use — Letting the LLM Call External APIs

So far, our chains only generate text. But what if the LLM needs **real-time data** it wasn't trained on — like the current weather?

With **tool use**, we define Python functions as "tools" and let the LLM decide when to call them. This is the foundation of AI agents (the ReAct pattern: Reason → Act → Observe).

We'll use the free [Open-Meteo API](https://open-meteo.com/) — no API key needed.

### 4d: Full Workflow with Error Handling

Let's put it all together — a robust workflow that handles errors at each step.

In [ ]:
def run_robust_workflow(topic):
    """Run the content generation workflow with error handling at each step."""
    results = {"topic": topic, "errors": []}

    # Step 1: Generate outline
    print(f"Processing topic: {topic}")
    print("-" * 40)

    try:
        print("Step 1: Generating outline...")
        outline = call_with_retry(outline_chain, {"topic": topic})
        results["outline"] = outline
        print("  Outline generated.")
    except Exception as e:
        results["errors"].append(f"Outline generation failed: {e}")
        print(f"  FAILED: {e}")
        # Can't continue without an outline
        return results

    # Step 2: Write draft
    try:
        print("Step 2: Writing draft...")
        # Validate the outline isn't empty or too short
        if len(outline.strip()) < 50:
            raise ValueError("Outline is too short — LLM may have returned garbage.")
        draft = call_with_retry(draft_chain, {"outline": outline})
        results["draft"] = draft
        print("  Draft written.")
    except Exception as e:
        results["errors"].append(f"Draft generation failed: {e}")
        print(f"  FAILED: {e}")
        return results

    # Step 3: Summarize
    try:
        print("Step 3: Summarizing...")
        summary = call_with_retry(summary_chain, {"draft": draft})
        results["summary"] = summary
        print("  Summary created.")
    except Exception as e:
        results["errors"].append(f"Summary generation failed: {e}")
        print(f"  FAILED (non-critical): {e}")
        # Summary is optional — we still have the draft
        results["summary"] = "(Summary unavailable due to error)"

    print("-" * 40)
    if results["errors"]:
        print(f"Completed with {len(results['errors'])} error(s).")
    else:
        print("Completed successfully!")

    return results


# Run the robust workflow
results = run_robust_workflow("building REST APIs with Python")
print("\n" + "=" * 60)
print("FINAL SUMMARY:")
print(results.get("summary", "No summary available"))

## Your Turn!

Modify the workflow above or create a new one. Some ideas:

- Change the prompts to generate different types of content
- Add a step that translates the summary to another language
- Add JSON output validation to one of the steps
- Create a branching workflow where the LLM decides which path to take

Remember: your final project should have **at least 3 steps** and include **error handling**.

In [ ]:
# YOUR WORKFLOW HERE
# Start experimenting!


In [ ]:
import requests
from langchain_core.tools import tool


# Define a tool — a regular Python function decorated with @tool
# The docstring is critical: the LLM reads it to decide WHEN to use the tool
@tool
def get_current_weather(latitude: float, longitude: float) -> str:
    """Get the current weather for a location given its latitude and longitude.

    Use this tool when someone asks about current weather conditions.
    Common coordinates:
    - New York City: 40.71, -74.01
    - Los Angeles: 34.05, -118.24
    - London: 51.51, -0.13
    - Tokyo: 35.68, 139.69
    - Troy, NY: 42.73, -73.69
    """
    try:
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,wind_speed_10m,weather_code",
            "temperature_unit": "fahrenheit",
            "wind_speed_unit": "mph",
        }
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        current = data["current"]

        # Map weather codes to descriptions
        weather_codes = {
            0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy",
            3: "Overcast", 45: "Foggy", 51: "Light drizzle",
            61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
            71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
            95: "Thunderstorm",
        }
        condition = weather_codes.get(current["weather_code"], "Unknown")

        return (
            f"Temperature: {current['temperature_2m']}°F, "
            f"Wind: {current['wind_speed_10m']} mph, "
            f"Conditions: {condition}"
        )
    except Exception as e:
        return f"Error fetching weather: {e}"


# Test the tool directly (no LLM involved)
print("Direct tool call (Troy, NY):")
print(get_current_weather.invoke({"latitude": 42.73, "longitude": -73.69}))

In [ ]:
# Bind the tool to the LLM — now the LLM knows this tool exists
# and can decide to call it when appropriate
llm_with_tools = llm.bind_tools([get_current_weather])

# Ask a question that requires the tool
print("Asking the LLM about weather (it will decide to use the tool)...\n")
response = llm_with_tools.invoke("What's the weather like right now in Troy, NY?")

# The LLM doesn't return the weather directly — it returns a "tool call"
# telling us WHICH tool to call and WITH WHAT arguments
print("LLM response type:", type(response).__name__)
print("Content:", response.content[:100] if response.content else "(empty — tool call instead)")
print()

if response.tool_calls:
    print("Tool calls the LLM wants to make:")
    for tc in response.tool_calls:
        print(f"  Function: {tc['name']}")
        print(f"  Arguments: {tc['args']}")
else:
    print("No tool calls — LLM answered directly.")

**What just happened?** The LLM didn't fetch the weather itself — it told us it *wants* to call `get_current_weather` with specific coordinates. We need to:
1. Execute the tool call ourselves
2. Send the result back to the LLM
3. Let the LLM compose a natural language answer

This is the **tool use loop** — the core of AI agents. Let's automate it:

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

def ask_with_tools(question):
    """Complete tool-use loop: ask → LLM decides tool call → execute → LLM answers."""
    tools = [get_current_weather]
    tools_by_name = {t.name: t for t in tools}
    llm_with_tools = llm.bind_tools(tools)

    # Step 1: Send the question to the LLM
    messages = [HumanMessage(content=question)]
    ai_response = llm_with_tools.invoke(messages)
    messages.append(ai_response)

    # Step 2: If the LLM wants to call tools, execute them
    if ai_response.tool_calls:
        for tool_call in ai_response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            # Guard: LLMs sometimes hallucinate tool names that don't exist
            if tool_name not in tools_by_name:
                print(f"  LLM tried to call unknown tool '{tool_name}' — skipping.")
                messages.append(ToolMessage(
                    content=f"Error: tool '{tool_name}' does not exist. "
                            f"Available tools: {list(tools_by_name.keys())}",
                    tool_call_id=tool_call["id"]
                ))
                continue

            print(f"  Calling tool: {tool_name}({tool_args})")

            # Execute the tool
            tool_fn = tools_by_name[tool_name]
            result = tool_fn.invoke(tool_args)
            print(f"  Tool result: {result}\n")

            # Send the result back to the LLM as a ToolMessage
            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            ))

        # Step 3: LLM composes a final answer using the tool results
        final_response = llm_with_tools.invoke(messages)
        return final_response.content
    else:
        # No tool call needed — LLM answered directly
        return ai_response.content


# Try it out!
print("=" * 60)
print("Question: What's the weather in Troy, NY right now?")
print("=" * 60)
answer = ask_with_tools("What's the weather in Troy, NY right now? Give a brief summary.")
print(f"\nFinal answer:\n{answer}")

print("\n" + "=" * 60)
print("Question: Compare weather in two cities")
print("=" * 60)
answer2 = ask_with_tools(
    "What's the weather like in New York City vs Los Angeles right now? "
    "Which city is warmer?"
)
print(f"\nFinal answer:\n{answer2}")

**Key takeaways from tool use:**
- The `@tool` decorator turns any Python function into a LangChain tool
- The **docstring** tells the LLM when and how to use the tool — write it carefully
- The LLM **decides** whether to use a tool and can call it **multiple times** (e.g., weather for two cities)
- This is a **workflow**: Question → LLM decides → Tool executes → LLM summarizes
- **LLMs can hallucinate tool names** — our code guards against this by checking `tools_by_name` before calling. This is a real-world error you must handle!
- Error handling matters at every level: the tool catches API errors, and the loop catches unknown tool names
- This pattern scales — you can bind multiple tools and the LLM picks the right one